# Итоговый проект: популярность и активность Python-репозиториев GitHub

**Курс:** Автоматизированный сбор данных из интернета (код + no-code)
**Автор:** Подольский Ф. А., ПМИ-3-2025 · **Конфигурация:** B (гибрид: код собирает и обрабатывает, n8n оркестрирует)

**Исследовательский вопрос:** *Как популярность и активность Python-репозиториев связаны с их тематикой и размером?*

Блокнот построен по структуре шаблона итогового проекта (разделы 0–16). Код пайплайна перенесён в модули `src/`
(раздел 16), поэтому здесь он **не дублируется**: блокнот импортирует модули, показывает доказательства по каждому
требованию и строит аналитику по слою `mart/`.

**Порядок работы.** Сначала выполните пайплайн (`./run_real.sh` или `python src/run_pipeline.py`), затем
`Kernel → Restart & Run All`. Блокнот исполняется целиком в любой момент: если данных ещё нет, ячейки сообщают,
какой шаг нужно выполнить, а не падают.

> Все выводы в разделе 8 формулируются **из вычислений** над `mart/`. Если данные получены на локальной имитации
> (`demo/mock_github.py`), блокнот помечает их как синтетические, и выводы по ним делать нельзя.

## 0. Паспорт проекта

Первый слайд защиты.

In [1]:
PROJECT_CARD = {
    "автор": "Подольский Ф. А., ПМИ-3-2025",
    "вариант_конфигурации": "B (гибрид): Python собирает и обрабатывает, n8n — расписание, запуск, оповещения",
    "исследовательский_вопрос": "Как популярность и активность Python-репозиториев связаны с их тематикой и размером?",
    "почему_нужны_два_источника": ("HTML-страницы тем GitHub дают тематическую принадлежность и порядок выдачи; "
                                   "REST API — метрики популярности (звёзды, форки, наблюдатели), активности "
                                   "(pushed_at, open issues) и размера (size). Ни один источник отдельно вопрос не закрывает: "
                                   "связь — по ключу owner/repo."),
    "ожидаемый_объём_записей": 560,      # 6 тем x 5 страниц x 20 карточек = 600 карточек, ~560 уникальных репозиториев
    "частота_обновления": "ежедневно в 03:17 UTC (Schedule Trigger в n8n)",
    "контакт_в_user_agent": "PSU-edu-crawler/1.0 (+https://github.com/Meta1rus/web-data-extraction; student lab)",
    "есть_персональные_данные": False,   # только логин владельца как часть публичного ключа owner/repo; см. docs/legal_checklist.md
}

def check_card(card):
    problems = [k for k, v in card.items() if v is None or (isinstance(v, str) and v.startswith("TODO"))]
    print("Не заполнено: " + ", ".join(problems) if problems else "Паспорт проекта заполнен.")
    return not problems

CARD_OK = check_card(PROJECT_CARD)

Паспорт проекта заполнен.


## 1. Окружение, конфигурация и служебная инфраструктура

In [2]:
import json, sqlite3, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src" / "run_pipeline.py").exists():          # блокнот открыт не из корня проекта
    ROOT = Path(__file__).resolve().parent if "__file__" in globals() else ROOT
sys.path.insert(0, str(ROOT / "src"))

from settings import DIRS, load_config, mask_secret, github_token
import settings, http_client, collect_html, collect_api, clean, contract, storage, quality, observability

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

CFG = load_config(ROOT / "config.yaml")
DB_PATH = settings.DATA_ROOT / CFG["storage"]["db_path"]


def rel(p):
    for base in (settings.DATA_ROOT, ROOT):
        try:
            return str(Path(p).relative_to(base))
        except ValueError:
            continue
    return str(p)
USER_AGENT = CFG["http"]["user_agent"]
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| matplotlib:", matplotlib.__version__)
print("Корень проекта:", ROOT)
print("Каталог данных:", settings.DATA_ROOT, "" if settings.DATA_ROOT == ROOT.resolve() else "(PIPELINE_DATA_ROOT)")
print("User-Agent:", USER_AGENT)
print("GITHUB_TOKEN:", mask_secret(github_token()))
print("БД:", DB_PATH, "(есть)" if DB_PATH.exists() else "(ещё нет — запустите пайплайн)")


def db():
    return sqlite3.connect(DB_PATH) if DB_PATH.exists() else None


def latest_json(pattern):
    files = sorted(DIRS["reports"].glob(pattern))
    return json.loads(files[-1].read_text(encoding="utf-8")) if files else None


# Происхождение данных: реальный GitHub или локальная имитация (адреса записаны в журнале прогонов)
DATA_ORIGIN = "нет данных"
if DB_PATH.exists():
    with sqlite3.connect(DB_PATH) as c:
        bases = c.execute("SELECT DISTINCT html_base, api_base FROM pipeline_runs").fetchall()
    if bases:
        local = any("127.0.0.1" in h or "localhost" in h or "127.0.0.1" in a for h, a in bases)
        DATA_ORIGIN = "ИМИТАЦИЯ (синтетические данные)" if local else "реальный GitHub"
print("Происхождение данных:", DATA_ORIGIN)
if DATA_ORIGIN.startswith("ИМИТАЦИЯ"):
    display(Markdown("> **Внимание: данные получены на локальной имитации `demo/mock_github.py`. "
                     "Цифры ниже проверяют работу кода, но выводами исследования не являются.**"))

Python: 3.10.12 | pandas: 2.3.3 | matplotlib: 3.10.9
Корень проекта: /sessions/rcw-012ewfrivctrzzfvfwmihxbt/run/final
Каталог данных: /sessions/rcw-012ewfrivctrzzfvfwmihxbt/run/final 
User-Agent: PSU-edu-crawler/1.0 (+https://github.com/Meta1rus/web-data-extraction; student lab)
GITHUB_TOKEN: gith…xD (длина 93)
БД: /sessions/rcw-012ewfrivctrzzfvfwmihxbt/run/final/db/warehouse.db (есть)
Происхождение данных: реальный GitHub


### 1.1. Конфигурация источников (паспорта)

Требование курса: минимум два источника — **один HTML-сайт и один REST API**; оба из списка разрешённых (связка № 5).

In [3]:
h, a = CFG["sources"]["html"], CFG["sources"]["api"]
SOURCES = {
    "html_catalog": {
        "тип": "html",
        "название": "GitHub Topics — github.com/topics/<topic>",
        "точка_входа": f"{h['base_url']}/topics/<topic>?l={h['language_filter']}&page=N",
        "пагинация": "кнопка «Load more…» = GET той же страницы с page=N+1; 20 карточек на странице; стоп — нет карточек или нет ссылки на следующую страницу",
        "нужен_браузер": False,
        "ключ_уникальности": "owner/repo из ссылки карточки (в нижнем регистре)",
        "лимит_запросов_в_сек": h["rate_per_sec"],
        "персональные_данные": "логин владельца в составе owner/repo (публичный псевдоним); аватары и профили не собираются",
        "условия_использования": "robots.txt разрешает /topics/* для User-agent: *; GitHub AUP разд. 7 — исследования на публичных неперсональных данных при открытом доступе к публикациям",
        "формат_ответа": "HTML (gzip)",
    },
    "api_source": {
        "тип": "api",
        "название": "GitHub REST API",
        "точка_входа": f"{a['base_url']}/repos/{{owner}}/{{repo}}; без токена — /search/repositories",
        "пагинация": "поштучные запросы по ключам из HTML; для Search — заголовок Link rel=next, per_page=100",
        "лимиты_из_документации": "5000 запр/ч с токеном, 60 запр/ч без токена; Search — 30/10 запр/мин; вторичные лимиты (429/403 + Retry-After)",
        "ключ_уникальности": "full_name -> repo_key (owner/repo, нижний регистр); repo_id — стабильный суррогат при переименовании",
        "лимит_запросов_в_сек": a["rate_per_sec"],
        "персональные_данные": "owner.login, owner.type; прочие поля владельца отбрасываются при сборе (minimize)",
        "условия_использования": "GitHub Terms of Service и API Terms; соблюдение лимитов, честный User-Agent, токен — только из окружения",
        "формат_ответа": "JSON (application/vnd.github+json, X-GitHub-Api-Version: 2022-11-28)",
    },
}
pd.DataFrame(SOURCES).fillna("—")

,html_catalog,api_source
тип,html,api
название,GitHub Topics — github.com/topics/<topic>,GitHub REST API
точка_входа,https://github.com/topics/<topic>?l=python&page=N,https://api.github.com/repos/{owner}/{repo}; без токена — /search/repositories
пагинация,кнопка «Load more…» = GET той же страницы с page=N+1; 20 карточек на страниц...,"поштучные запросы по ключам из HTML; для Search — заголовок Link rel=next, p..."
нужен_браузер,False,—
ключ_уникальности,owner/repo из ссылки карточки (в нижнем регистре),"full_name -> repo_key (owner/repo, нижний регистр); repo_id — стабильный сур..."
лимит_запросов_в_сек,0.33,1.5
персональные_данные,логин владельца в составе owner/repo (публичный псевдоним); аватары и профил...,"owner.login, owner.type; прочие поля владельца отбрасываются при сборе (mini..."
условия_использования,robots.txt разрешает /topics/* для User-agent: *; GitHub AUP разд. 7 — иссле...,"GitHub Terms of Service и API Terms; соблюдение лимитов, честный User-Agent,..."
формат_ответа,HTML (gzip),"JSON (application/vnd.github+json, X-GitHub-Api-Version: 2022-11-28)"


## 2. Правовая и техническая проверка источников

### 2.1. robots.txt и Crawl-delay

In [4]:
from http_client import RobotsRules
from urllib import robotparser

saved = sorted((DIRS["raw"] / "robots").glob("*.txt")) if (DIRS["raw"] / "robots").exists() else []
real = [p for p in saved if not p.name.startswith("127.0.0.1")]
robots_file = (real or saved or [ROOT / "tests/fixtures/github_robots.txt"])[-1]
print("Файл robots.txt:", rel(robots_file),
      "" if real else "(копия из прогона на имитации или фикстура — после реального прогона здесь будет github.com_<дата>.txt)")
text = robots_file.read_text(encoding="utf-8")
rules = RobotsRules(text, USER_AGENT)
rp = robotparser.RobotFileParser(); rp.parse(text.splitlines())
probe = ["/topics/web-scraping?l=python&page=2", "/topics/cli", "/scrapy/scrapy",
         "/scrapy/scrapy/stargazers", "/scrapy/scrapy/contributors", "/scrapy/scrapy?tab=readme-ov-file",
         "/search?q=scrapy", "/scrapy/scrapy/tree/master"]
rows = []
for p in probe:
    ok, rule = rules.can_fetch("https://github.com" + p)
    rows.append({"путь": p, "RFC 9309 (решение проекта)": "можно" if ok else "нельзя", "правило": rule or "—",
                 f"urllib.robotparser (Python {sys.version.split()[0]})": "можно" if rp.can_fetch(USER_AGENT, "https://github.com" + p) else "нельзя"})
print(f"Группа для нашего User-Agent: {rules.group_agent!r}; Crawl-delay: {rules.crawl_delay}; Sitemap: {rules.sitemaps or 'нет'}")
display(pd.DataFrame(rows))
print("Вывод: страницы тем разрешены; карточки репозиториев и их подстраницы (stargazers, contributors, ?tab=) не запрашиваются.")
print("urllib.robotparser в Python 3.10–3.12 не понимает шаблоны * и $ (ложное «можно» для /*/*/stargazers),")
print("поэтому решение принимает собственный матчер RFC 9309 (src/http_client.py::RobotsRules).")

Файл robots.txt: raw/robots/github.com_20261015.txt 
Группа для нашего User-Agent: '*'; Crawl-delay: None; Sitemap: нет


,путь,RFC 9309 (решение проекта),правило,urllib.robotparser (Python 3.10.12)
0,/topics/web-scraping?l=python&page=2,можно,—,можно
1,/topics/cli,можно,—,можно
2,/scrapy/scrapy,можно,—,можно
3,/scrapy/scrapy/stargazers,нельзя,disallow: /*/*/stargazers,можно
4,/scrapy/scrapy/contributors,нельзя,disallow: /*/*/contributors,можно
5,/scrapy/scrapy?tab=readme-ov-file,нельзя,disallow: /*?tab=*,можно
6,/search?q=scrapy,нельзя,disallow: /*?q=*,можно
7,/scrapy/scrapy/tree/master,нельзя,disallow: /*/tree/,можно


Вывод: страницы тем разрешены; карточки репозиториев и их подстраницы (stargazers, contributors, ?tab=) не запрашиваются.
urllib.robotparser в Python 3.10–3.12 не понимает шаблоны * и $ (ложное «можно» для /*/*/stargazers),
поэтому решение принимает собственный матчер RFC 9309 (src/http_client.py::RobotsRules).


### 2.2. Нужен ли браузер

Протокол занятия 4 (до выбора Playwright): 1) запросить страницу `requests` и найти текст; 2) DevTools → Network → XHR;
3) найденный эндпоинт — собирать из него; 4) браузер — только если данных нет ни в HTML, ни в API.
Диагностика выполняется командой `python src/collect_html.py --diagnose` и сохраняется в `reports/diagnostics_latest.json`.

In [5]:
diag_path = DIRS["reports"] / "diagnostics_latest.json"
if diag_path.exists():
    diag = json.loads(diag_path.read_text(encoding="utf-8"))
    display(pd.Series({k: v for k, v in diag.items() if k != "robots"}, name="значение").to_frame())
else:
    # офлайн-вариант: тот же протокол на последнем сохранённом снимке страницы из сырого слоя
    snaps = sorted((DIRS["raw"] / "html_pages").rglob("p01_*.html.gz")) if (DIRS["raw"] / "html_pages").exists() else []
    if snaps:
        import gzip
        html = gzip.decompress(snaps[-1].read_bytes()).decode("utf-8")
        cards, meta = collect_html.parse_topic_page(html, snaps[-1].parent.name, 1)
        needle = cards[0]["full_name"] if cards else "?"
        display(pd.Series({"снимок": rel(snaps[-1]), "байт": len(html), "карточек (requests+bs4)": len(cards),
                           "стратегия": meta["strategy"], "искомый текст": needle, "текст в HTML": needle in html,
                           "форма Load more": "ajax-pagination-form" in html,
                           "вывод": "достаточно HTTP" if cards else "нужна ручная проверка"}, name="значение").to_frame())
    else:
        print("Нет ни отчёта диагностики, ни снимков страниц. Выполните: python src/collect_html.py --diagnose")
print("\nРучная часть протокола (скриншоты docs/screenshot_view_source.png, docs/screenshot_network.png — сделать при реальном прогоне):")
print("  View Source (Ctrl+U) -> Ctrl+F по имени первого репозитория -> найдено в исходном HTML;")
print("  Network -> нажать «Load more…» -> запрос GET /topics/<topic>?l=python&page=2 возвращает HTML-фрагмент с карточками.")

,значение
снимок,raw/html_pages/web-scraping/p01_20261015T215039Z.html.gz
байт,642428
карточек (requests+bs4),20
стратегия,article-cards
искомый текст,browser-use/browser-use
текст в HTML,True
форма Load more,True
вывод,достаточно HTTP



Ручная часть протокола (скриншоты docs/screenshot_view_source.png, docs/screenshot_network.png — сделать при реальном прогоне):
  View Source (Ctrl+U) -> Ctrl+F по имени первого репозитория -> найдено в исходном HTML;
  Network -> нажать «Load more…» -> запрос GET /topics/<topic>?l=python&page=2 возвращает HTML-фрагмент с карточками.


## 3. Вежливый и устойчивый HTTP-клиент

Реализация — `src/http_client.py` (единая точка выхода в сеть `PoliteClient.get`). Ниже — офлайн-демонстрации
механизмов без обращения к сайтам: лимит на домен с джиттером, классификация ошибок, backoff, предохранитель.

In [6]:
from http_client import RateLimiter, classify, backoff_delay, CircuitBreaker, server_requested_wait

limits = pd.DataFrame([
    {"ключ лимита": "html (github.com)", "запр/с": CFG["sources"]["html"]["rate_per_sec"], "всплеск": CFG["sources"]["html"]["burst"]},
    {"ключ лимита": "api (api.github.com)", "запр/с": CFG["sources"]["api"]["rate_per_sec"], "всплеск": CFG["sources"]["api"]["burst"]},
    {"ключ лимита": "api_search (/search)", "запр/с": CFG["sources"]["api"]["search_rate_per_sec"], "всплеск": 1},
])
limits["интервал, с"] = (1 / limits["запр/с"]).round(2)
display(limits)
print("Джиттер:", CFG["http"]["jitter"], "| Crawl-delay из robots.txt, если задан, имеет приоритет (берётся меньшая скорость).")

lim = RateLimiter(rate=5, burst=1, jitter=0.3)
marks = []
for _ in range(6):
    lim.acquire(); marks.append(time.monotonic())
gaps = np.diff(marks).round(3)
print("Интервалы при лимите 5 запр/с (ожидаемо ~0.2 c + джиттер до 0.06 c):", list(gaps))

table = [{"код/ситуация": k, "класс": classify(k)[0], "повторять": "да" if classify(k)[1] else "нет"}
         for k in (200, 304, 400, 401, 403, 404, 429, 500, 503)]
table.append({"код/ситуация": "403 + X-RateLimit-Remaining: 0", "класс": classify(403, headers={"X-RateLimit-Remaining": "0"})[0], "повторять": "после X-RateLimit-Reset"})
table.append({"код/ситуация": "таймаут", "класс": "timeout", "повторять": "да"})
display(pd.DataFrame(table))
print("Retry-After важнее backoff:", server_requested_wait({"Retry-After": "7"}, "rate_limited"))
print("Backoff full jitter, верхние границы по попыткам:", [min(30, 2 ** a) for a in range(6)],
      "| пример:", [round(backoff_delay(a), 2) for a in range(6)])

cb = CircuitBreaker(fail_threshold=3, reset_after_s=0.5)
timeline = []
for i, ok in enumerate([False, False, False, None, None, True, True]):
    allowed = cb.allow()
    if not allowed:
        timeline.append({"шаг": i + 1, "состояние": cb.state, "запрос": "пропущен"}); time.sleep(0.3); continue
    before = cb.state
    cb.record(bool(ok) if ok is not None else True)
    timeline.append({"шаг": i + 1, "состояние": before, "запрос": "успех" if ok in (True, None) else "отказ", "после": cb.state})
display(pd.DataFrame(timeline))

,ключ лимита,запр/с,всплеск,"интервал, с"
0,html (github.com),0.33,1,3.03
1,api (api.github.com),1.50,3,0.67
2,api_search (/search),0.15,1,6.67


Джиттер: 0.3 | Crawl-delay из robots.txt, если задан, имеет приоритет (берётся меньшая скорость).


Интервалы при лимите 5 запр/с (ожидаемо ~0.2 c + джиттер до 0.06 c): [np.float64(0.178), np.float64(0.202), np.float64(0.199), np.float64(0.234), np.float64(0.16)]


,код/ситуация,класс,повторять
0,200,ok,нет
1,304,not_modified,нет
2,400,client_error,нет
3,401,unauthorized,нет
4,403,forbidden,нет
5,404,not_found,нет
6,429,rate_limited,да
7,500,server_error,да
8,503,server_error,да
9,403 + X-RateLimit-Remaining: 0,rate_limit_primary,после X-RateLimit-Reset


Retry-After важнее backoff: (7.0, 'Retry-After')
Backoff full jitter, верхние границы по попыткам: [1, 2, 4, 8, 16, 30] | пример: [0.91, 0.39, 1.11, 4.11, 1.73, 2.68]


,шаг,состояние,запрос,после
0,1,closed,отказ,closed
1,2,closed,отказ,closed
2,3,closed,отказ,open
3,4,open,пропущен,NaN
4,5,open,пропущен,NaN
5,6,half-open,успех,closed
6,7,closed,успех,closed


## 4. Сбор: сырой слой

Сырой слой пишется **только дозаписью** с метаданными происхождения (`_run_id`, `_source`, `_collected_at`, `_url`, `_status`, `_parser_version`, `_html_sha256`, `_etag`). Страницы тем целиком сохраняются в `raw/html_pages/` (gzip) — для перепарсинга без повторного обращения к сайту.

In [7]:
from collect_html import read_raw
raw_html = read_raw(collect_html.SOURCE)
raw_api = read_raw(collect_api.SOURCE)
if not raw_html and not raw_api:
    print("Сырой слой пуст — запустите пайплайн.")
else:
    summary = (pd.DataFrame([{"источник": e["_source"], "прогон": e["_run_id"]} for e in raw_html + raw_api])
               .value_counts().rename("строк").reset_index().sort_values(["прогон", "источник"]))
    display(summary)
    print("Пример конверта сырого слоя (HTML):")
    print(json.dumps({k: v for k, v in raw_html[0].items() if k != "payload"}, ensure_ascii=False, indent=1)[:900] if raw_html else "—")
    for name in ("github_topics_html.jsonl", "github_api_repos.jsonl"):
        p = DIRS["raw"] / name
        if p.exists():
            print(f"{name}: {p.stat().st_size / 1024:.0f} КБ")
    snaps = list((DIRS["raw"] / "html_pages").rglob("*.html.gz"))
    print("Снимков HTML-страниц:", len(snaps))

,источник,прогон,строк
7,github_api_repos,20261015T212239Z,91
1,github_topics_html,20261015T212239Z,600
4,github_api_repos,20261015T212517Z,254
5,github_api_repos,20261015T212811Z,247
14,github_api_repos,20261015T213102Z,10
2,github_topics_html,20261015T213102Z,600
6,github_api_repos,20261015T213248Z,113
0,github_topics_html,20261015T213248Z,600
3,github_api_repos,20261015T214236Z,255
13,github_api_repos,20261015T214611Z,16


Пример конверта сырого слоя (HTML):
{
 "_run_id": "20261015T212239Z",
 "_source": "github_topics_html",
 "_collected_at": "2026-10-15T21:22:42Z",
 "_url": "https://github.com/topics/web-scraping?l=python",
 "_status": 200,
 "_parser_version": "gh-topics-1.0.0",
 "_html_sha256": "e681aa12b4fba2a7b27bc3764b604a21a1ff16bd1a01983c53486dd59980be41",
 "_snapshot": "raw/html_pages/web-scraping/p01_20261015T212239Z.html.gz",
 "_strategy": "article-cards",
 "topic": "web-scraping",
 "page": 1,
 "sort_mode": "stars"
}
github_topics_html.jsonl: 1992 КБ
github_api_repos.jsonl: 1400 КБ
Снимков HTML-страниц: 97


## 5. Очистка и контракт данных

Протокол очистки (`clean/cleaning_protocol.json`) пишется пайплайном на каждом прогоне; контракт — `src/contract.py` (pydantic, по 4–5 собственных валидаторов на сущность).

In [8]:
prot = DIRS["clean"] / "cleaning_protocol.json"
if prot.exists():
    display(pd.DataFrame(json.loads(prot.read_text(encoding="utf-8"))["steps"]))
rej_path = DIRS["clean"] / "rejects.jsonl"
rejects = [json.loads(l) for l in rej_path.read_text(encoding="utf-8").splitlines() if l.strip()] if rej_path.exists() else []
print("Отбраковано записей:", len(rejects))
if rejects:
    display(pd.DataFrame(rejects)[["source", "record_key", "field", "reason"]].head(15))

,шаг,до,после,затронуто,примечание
0,1. чтение raw: карточки тем,1940,1940,0,прогонов: 8
1,1. чтение raw: ответы API,993,993,0,прогонов: 10
2,"2–5. нормализация текста, ключ, типы (карточки)",1940,1940,0,описаний с исправленными пробелами/символами: 83
3,"2–5. нормализация текста, ключ, типы (API)",993,993,0,
4,"6. дедупликация карточек по (topic, repo_key)",1940,600,1340,повторные наблюдения тех же карточек в разных прогонах; оставлена самая свежая
5,6. дедупликация API по repo_key и repo_id,993,592,401,переименований/синонимов ключа: 0
6,7. контракт: карточки тем,600,600,0,в rejects: 0
7,7. контракт: репозитории API,592,592,0,в rejects: 0
8,8. вычисляемые поля,592,592,0,"age_days, days_since_push, stars_per_year, size_mb, size_band, activity_band..."


Отбраковано записей: 0


In [9]:
# Контракт ловит то, что прошло сквозь очистку: проверка на заведомо битых записях
from datetime import datetime, timezone, timedelta
from contract import RepoRecord, TopicListing, validate_records

base = {"repo_key": "fx-owner/repo", "repo_id": 1, "full_name": "fx-owner/repo", "owner_login": "fx-owner",
        "owner_type": "User", "stars": 10, "forks": 1, "open_issues": 0, "size_kb": 100, "is_archived": False,
        "is_fork": False, "created_at": datetime(2020, 1, 1, tzinfo=timezone.utc),
        "gh_updated_at": datetime(2026, 1, 1, tzinfo=timezone.utc), "collected_at": datetime(2026, 10, 15, tzinfo=timezone.utc),
        "source": "github_rest_api", "parser_version": "t", "run_id": "probe", "topics": ["python"]}
probes = [("отрицательные звёзды", {"stars": -1}), ("пробел в имени владельца", {"full_name": "bad owner/repo", "repo_key": "bad owner/repo"}),
          ("тема с заглавными и «!»", {"topics": ["Web Scraping!"]}), ("created_at в будущем", {"created_at": datetime(2030, 1, 1, tzinfo=timezone.utc)}),
          ("дата без часового пояса", {"created_at": datetime(2020, 1, 1)}), ("ключ не совпадает с full_name", {"repo_key": "x/y"}),
          ("updated_at раньше created_at", {"gh_updated_at": datetime(2019, 1, 1, tzinfo=timezone.utc)})]
rows = []
for name, patch in probes:
    v, r = validate_records([{**base, **patch}], RepoRecord, "probe")
    rows.append({"проверка": name, "результат": "ПРОПУЩЕНА (плохо)" if v else "отклонена",
                 "поле": r[0]["field"] if r else "", "сообщение": r[0]["reason"][:90] if r else ""})
display(pd.DataFrame(rows))

,проверка,результат,поле,сообщение
0,отрицательные звёзды,отклонена,stars,Input should be greater than or equal to 0
1,пробел в имени владельца,отклонена,full_name,"Value error, недопустимое имя владельца: 'bad owner'"
2,тема с заглавными и «!»,отклонена,topics,"Value error, недопустимая тема 'web scraping!': только a-z, 0-9 и дефис, до ..."
3,created_at в будущем,отклонена,__record__,"Value error, created_at позже момента сбора — ошибка часового пояса или разбора"
4,дата без часового пояса,отклонена,created_at,"Value error, дата без часового пояса недопустима (ожидается UTC)"
5,ключ не совпадает с full_name,отклонена,__record__,"Value error, repo_key должен совпадать с full_name в нижнем регистре"
6,updated_at раньше created_at,отклонена,__record__,"Value error, updated_at раньше created_at"


## 6. Хранилище и идемпотентная загрузка

Схема — `db/schema.sql`: PK по бизнес-ключу, NOT NULL, CHECK, индексы, служебные поля. UPSERT с условием `WHERE row_hash IS NOT excluded.row_hash`. Демонстрация тремя прогонами — таблица `pipeline_runs`.

In [10]:
conn = db()
if conn is None:
    print("БД ещё нет — запустите пайплайн.")
else:
    tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%'")]
    print("Таблицы и представления:", tables)
    display(pd.read_sql_query("SELECT 'repos' t, COUNT(*) n FROM repos UNION ALL SELECT 'topic_listings', COUNT(*) FROM topic_listings "
                              "UNION ALL SELECT 'repo_snapshots', COUNT(*) FROM repo_snapshots UNION ALL SELECT 'rejects', COUNT(*) FROM rejects", conn))
    runs = pd.read_sql_query("SELECT run_id, status, verdict, exit_code, html_cards, api_200, api_304, inserted, updated, "
                             "unchanged, clean_repos, mart_built FROM pipeline_runs ORDER BY started_at", conn)
    print("Прогоны (UPSERT: вставлено / обновлено / без изменений по таблице repos):")
    display(runs)
    dup = conn.execute("SELECT COUNT(*) FROM (SELECT repo_key FROM repos GROUP BY repo_key HAVING COUNT(*) > 1)").fetchone()[0]
    print("Дублей по бизнес-ключу в repos:", dup)

Таблицы и представления: ['repos', 'topic_listings', 'repo_aliases', 'repo_snapshots', 'rejects', 'pipeline_runs', 'v_repo_topic', 'v_unenriched_listings']


,t,n
0,repos,592
1,topic_listings,600
2,repo_snapshots,294
3,rejects,0


Прогоны (UPSERT: вставлено / обновлено / без изменений по таблице repos):


,run_id,status,verdict,exit_code,html_cards,api_200,api_304,inserted,updated,unchanged,clean_repos,mart_built
0,20261015T212239Z,interrupted,red,130,600,91,0,NaN,NaN,NaN,NaN,0
1,20261015T212517Z,interrupted,red,130,600,254,0,NaN,NaN,NaN,NaN,0
2,20261015T212811Z,finished,green,0,600,247,0,592.0,0.0,0.0,592.0,1
3,20261015T213102Z,finished,green,0,600,10,0,0.0,10.0,582.0,592.0,1
4,20261015T213248Z,interrupted,red,130,600,113,0,NaN,NaN,NaN,NaN,0
5,20261015T214236Z,interrupted,red,130,600,255,0,NaN,NaN,NaN,NaN,0
6,20261015T214611Z,finished,green,0,600,16,208,0.0,37.0,555.0,592.0,1
7,20261015T214857Z,finished,green,0,20,3,17,0.0,3.0,589.0,592.0,1
8,20261015T214919Z,finished,red,3,20,0,0,0.0,0.0,592.0,592.0,1
9,20261015T214925Z,interrupted,red,130,60,1,6,NaN,NaN,NaN,NaN,0


Дублей по бизнес-ключу в repos: 0


In [11]:
# Ограничения БД — второй рубеж: пробуем вставить заведомо плохие строки внутри транзакции и откатываем её
if conn is not None and conn.execute("SELECT COUNT(*) FROM repos").fetchone()[0]:
    row = dict(pd.read_sql_query("SELECT * FROM repos LIMIT 1", conn).iloc[0])
    tests = [("отрицательные звёзды (CHECK)", {"repo_key": "probe/one", "repo_id": 10**12, "stars": -5}),
             ("повтор первичного ключа", {"repo_id": 10**12 + 1}),
             ("ключ не в нижнем регистре (CHECK)", {"repo_key": "Probe/Two", "repo_id": 10**12 + 2}),
             ("is_archived = 7 (CHECK)", {"repo_key": "probe/three", "repo_id": 10**12 + 3, "is_archived": 7})]
    out = []
    for name, patch in tests:
        r = {k: (v.item() if hasattr(v, "item") else v) for k, v in {**row, **patch}.items()}
        cols = ", ".join(r); ph = ", ".join("?" * len(r))
        try:
            conn.execute("SAVEPOINT probe")
            conn.execute(f"INSERT INTO repos ({cols}) VALUES ({ph})", list(r.values()))
            out.append({"проверка": name, "результат": "вставлено (плохо)", "ошибка": ""})
        except sqlite3.IntegrityError as e:
            out.append({"проверка": name, "результат": "отклонено базой", "ошибка": str(e)[:70]})
        finally:
            conn.execute("ROLLBACK TO probe"); conn.execute("RELEASE probe")
    display(pd.DataFrame(out))

,проверка,результат,ошибка
0,отрицательные звёзды (CHECK),отклонено базой,CHECK constraint failed: stars >= 0
1,повтор первичного ключа,отклонено базой,UNIQUE constraint failed: repos.repo_key
2,ключ не в нижнем регистре (CHECK),отклонено базой,CHECK constraint failed: repo_key = lower(repo_key) AND instr(repo_key
3,is_archived = 7 (CHECK),отклонено базой,"CHECK constraint failed: is_archived IN (0, 1)"


## 7. Метрики качества и пороги

Пороги — `config.yaml → quality.metrics`; обоснование порогов — в `docs/decisions.md`, раздел 5. При КРАСНОМ статусе mart не обновляется, код возврата 2.

In [12]:
q = latest_json("quality_*.json")
if q is None:
    print("Отчёта о качестве ещё нет — запустите пайплайн.")
else:
    qdf = pd.DataFrame(q["metrics"])[["title", "value", "green", "yellow", "direction", "status"]]
    qdf.columns = ["метрика", "значение", "зелёный", "жёлтый", "лучше", "статус"]
    display(qdf)
    print("Вердикт:", q["verdict"].upper(), "| прогон", q["run_id"])
    print("Справочно:", q.get("info"))
    if q.get("reject_reasons"):
        display(pd.Series(q["reject_reasons"], name="записей").to_frame("записей"))

,метрика,значение,зелёный,жёлтый,лучше,статус
0,"полнота обязательных полей, %",100.00,99.5,98.0,higher,green
1,"доля отбракованных записей, %",0.00,2.0,5.0,lower,green
2,"уникальность бизнес-ключа, %",100.00,100.0,99.9,higher,green
3,"покрытие связи HTML -> API (обогащено), %",100.00,95.0,80.0,higher,green
4,"согласованность звёзд HTML и API, %",100.00,90.0,75.0,higher,green
5,"возраст данных API (p95), ч",0.37,48.0,168.0,lower,green
6,"объём чистого слоя к прошлому прогону, %",100.00,90.0,70.0,higher,green
7,"доля репозиториев на Python, %",100.00,85.0,70.0,higher,green
8,число обогащённых репозиториев для анализа,592.00,500.0,300.0,higher,green


Вердикт: GREEN | прогон 20261015T215039Z
Справочно: {'pushed_before_created_pct': 0.0, 'archived_pct': 7.94, 'forks_in_sample_pct': 0.51, 'stars_html_approx_pct': 0.0}


## 8. Слой аналитики (mart) и выводы

In [13]:
MART = {}
for name in ("repo_topic", "topic_summary", "size_band_topic", "activity_band_topic", "stars_by_day_topic"):
    p = DIRS["mart"] / f"{name}.parquet"
    if p.exists():
        MART[name] = pd.read_parquet(p)
if not MART:
    print("Слой mart пуст: пайплайн не запускался или качество было красным.")
else:
    rt = MART["repo_topic"]
    py = rt[rt["is_python"]].copy()
    print(f"Источник данных: {DATA_ORIGIN}; строк repo×topic: {len(rt)}, из них Python: {len(py)}; "
          f"уникальных Python-репозиториев: {py['repo_ref'].nunique()}")
    print("Период сбора (по API):", py["api_collected_at"].min(), "—", py["api_collected_at"].max())
    display(MART["topic_summary"])

Источник данных: реальный GitHub; строк repo×topic: 600, из них Python: 600; уникальных Python-репозиториев: 592
Период сбора (по API): 2026-10-15 21:28:27+00:00 — 2026-10-15 21:50:30+00:00


,topic,repos,stars_median,stars_p75,forks_median,size_mb_median,stars_per_year_median,active_90d_pct,archived_pct,days_since_push_median,issues_per_100_stars_median,spearman_size_stars,spearman_size_days_since_push
0,machine-learning,100,16549.0,29900.75,3060.5,103.94,2409.0,64.0,7.0,8.0,0.80,0.363,-0.425
1,cli,100,4694.0,8797.50,362.0,6.54,956.9,67.0,7.0,21.0,0.70,0.078,-0.526
2,fastapi,100,2236.0,4377.75,311.0,8.90,742.7,65.0,5.0,14.2,0.97,0.258,-0.463
3,telegram,100,831.0,1884.75,246.5,2.30,187.8,40.0,15.0,167.8,1.98,0.269,-0.234
4,web-scraping,100,738.0,2333.00,94.5,2.69,321.2,58.0,4.0,54.4,0.47,0.329,-0.367
5,data-engineering,100,428.0,1724.50,78.5,8.07,163.5,59.0,9.0,44.2,1.54,0.361,-0.511


In [14]:
# Вспомогательные вычисления: бутстреп-интервалы медиан и ранговая корреляция с перестановочным p
rng = np.random.default_rng(20261015)

def boot_median_ci(x, n=2000):
    x = np.asarray(x, dtype=float)
    meds = np.median(rng.choice(x, size=(n, len(x)), replace=True), axis=1)
    return np.percentile(meds, [2.5, 97.5])

def spearman(a, b):
    a, b = pd.Series(a).rank().to_numpy(), pd.Series(b).rank().to_numpy()
    return float(np.corrcoef(a, b)[0, 1])

def perm_p(a, b, n=2000):
    obs = abs(spearman(a, b)); a = np.asarray(a); b = np.asarray(b)
    hits = sum(abs(spearman(a, rng.permutation(b))) >= obs for _ in range(n))
    return (hits + 1) / (n + 1)

if MART:
    rows = []
    for t, g in py.groupby("topic"):
        lo, hi = boot_median_ci(g["stars"])
        rows.append({"тема": t, "n": len(g), "медиана звёзд": g["stars"].median(), "ДИ 95% низ": lo, "ДИ 95% верх": hi,
                     "медиана размера, МБ": round(g["size_mb"].median(), 2),
                     "ρ(размер, звёзды)": round(spearman(g["size_kb"], g["stars"]), 3),
                     "push ≤90 дн, %": round(100 * g["active_90d"].mean(), 1)})
    STATS = pd.DataFrame(rows).sort_values("медиана звёзд", ascending=False).reset_index(drop=True)
    display(STATS)

,тема,n,медиана звёзд,ДИ 95% низ,ДИ 95% верх,"медиана размера, МБ","ρ(размер, звёзды)","push ≤90 дн, %"
0,machine-learning,100,16549.0,14456.6500,19532.0,103.94,0.363,64.0
1,cli,100,4694.0,3580.0000,6213.0,6.54,0.078,67.0
2,fastapi,100,2236.0,1947.5000,2681.5,8.90,0.258,65.0
3,telegram,100,831.0,714.0000,1126.0,2.30,0.269,40.0
4,web-scraping,100,738.0,439.4125,1096.0,2.69,0.329,58.0
5,data-engineering,100,428.0,319.0000,652.5,8.07,0.361,59.0


### Графики

Цвета: одна серия — один нейтральный синий; для «доли» — последовательная шкала одного оттенка (светлее = меньше).
Подписи осей и заголовки обязательны; файлы сохраняются в `mart/`.

In [15]:
INK, MUTED, BLUE = "#0b0b0b", "#52514e", "#2a78d6"
plt.rcParams.update({"figure.dpi": 110, "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
SUFFIX = " — ИМИТАЦИЯ" if DATA_ORIGIN.startswith("ИМИТАЦИЯ") else ""

if MART:
    order = STATS["тема"].tolist()
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    data = [py.loc[py["topic"] == t, "stars"].to_numpy() for t in order]
    style = dict(widths=0.5, patch_artist=True, showfliers=False, medianprops={"color": INK, "linewidth": 1.6},
                 boxprops={"facecolor": "#d6e6f8", "edgecolor": BLUE, "linewidth": 1.2},
                 whiskerprops={"color": BLUE}, capprops={"color": BLUE})
    try:                                   # matplotlib >= 3.10
        ax.boxplot(data, orientation="horizontal", **style)
    except TypeError:                      # более старые версии
        ax.boxplot(data, vert=False, **style)
    ax.set_yticks(range(1, len(order) + 1), order)
    for i, d in enumerate(data, start=1):
        ax.scatter(d, np.full(len(d), i) + rng.uniform(-0.18, 0.18, len(d)), s=8, color=BLUE, alpha=0.35, linewidths=0)
        ax.text(np.max(d) * 1.15, i, f"медиана {np.median(d):,.0f} · n={len(d)}", va="center", fontsize=8, color=MUTED)
    ax.set_xscale("log")
    ax.set_xlim(right=ax.get_xlim()[1] * 6)
    ax.invert_yaxis()
    ax.set_xlabel("Звёзды (логарифмическая шкала)")
    ax.set_title("Популярность Python-репозиториев по темам GitHub" + SUFFIX, loc="left", color=INK)
    fig.tight_layout(); fig.savefig(DIRS["mart"] / "fig1_stars_by_topic.png", bbox_inches="tight"); plt.show()

In [16]:
if MART:
    topics = order
    fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True)
    for ax, t in zip(axes.ravel(), topics):
        g = py[py["topic"] == t]
        ax.scatter(g["size_mb"].clip(lower=0.01), g["stars"], s=10, color=BLUE, alpha=0.55, linewidths=0)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(f"{t}: ρ = {spearman(g['size_kb'], g['stars']):+.2f}, n={len(g)}", fontsize=9, loc="left", color=INK)
    for ax in axes[-1]:
        ax.set_xlabel("Размер репозитория, МБ (log)")
    for ax in axes[:, 0]:
        ax.set_ylabel("Звёзды (log)")
    fig.suptitle("Размер и популярность: связь внутри каждой темы (ρ — ранговая корреляция Спирмена)" + SUFFIX,
                 x=0.01, ha="left", color=INK)
    fig.tight_layout(); fig.savefig(DIRS["mart"] / "fig2_size_vs_stars.png", bbox_inches="tight"); plt.show()

In [17]:
if MART:
    band_order = ["<1 МБ", "1–10 МБ", "10–100 МБ", "≥100 МБ"]
    pv = (py.pivot_table(index="topic", columns="size_band", values="active_90d", aggfunc="mean") * 100).reindex(index=order, columns=band_order)
    cnt = py.pivot_table(index="topic", columns="size_band", values="repo_ref", aggfunc="count").reindex(index=order, columns=band_order)
    MIN_N = 5                                   # ячейки с n < 5 не окрашиваем: доля по 1–4 репозиториям ненадёжна
    shown = pv.where(cnt >= MIN_N)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    im = ax.imshow(shown.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(band_order)), band_order); ax.set_yticks(range(len(order)), order); ax.grid(False)
    for i in range(len(order)):
        for j in range(len(band_order)):
            v, n = pv.iat[i, j], cnt.iat[i, j]
            if pd.notna(v) and n >= MIN_N:
                ax.text(j, i, f"{v:.0f}%\nn={int(n)}", ha="center", va="center", fontsize=8, color="white" if v > 60 else INK)
            elif pd.notna(n) and n > 0:
                ax.text(j, i, f"n={int(n)}\nмало", ha="center", va="center", fontsize=7, color=MUTED)
    ax.set_xlabel("Размер репозитория"); ax.set_title("Доля репозиториев с push за последние 90 дней" + SUFFIX, loc="left", color=INK)
    fig.colorbar(im, ax=ax, label="% активных")
    fig.tight_layout(); fig.savefig(DIRS["mart"] / "fig3_activity_by_topic_size.png", bbox_inches="tight"); plt.show()

In [18]:
# Динамика за период сбора (снапшоты repo_snapshots): появляется после >= 2 дней прогонов
if MART and "stars_by_day_topic" in MART and MART["stars_by_day_topic"]["snapshot_date"].nunique() >= 2:
    d = MART["stars_by_day_topic"].pivot_table(index="snapshot_date", columns="topic", values="stars_total")
    idx = (d / d.iloc[0] * 100)
    fig, axes = plt.subplots(2, 3, figsize=(10, 5), sharex=True, sharey=True)
    for ax, t in zip(axes.ravel(), order):
        ax.plot(idx.index, idx[t], color=BLUE, linewidth=2); ax.set_title(t, fontsize=9, loc="left")
        ax.tick_params(axis="x", rotation=45)
    fig.suptitle("Сумма звёзд тем, индекс (первый день = 100)" + SUFFIX, x=0.01, ha="left")
    fig.tight_layout(); fig.savefig(DIRS["mart"] / "fig4_stars_dynamics.png", bbox_inches="tight"); plt.show()
else:
    print("Динамика: нужно не менее двух дат снапшотов (ежедневные прогоны).")

Динамика: нужно не менее двух дат снапшотов (ежедневные прогоны).


### Выводы

Формулировки строятся **из вычислений** ниже: утверждение подтверждается или ослабляется в зависимости от того,
пересекаются ли бутстреп-интервалы и значима ли корреляция. Для защиты достаточно 2–3 выводов.

In [19]:
FINDINGS, LIMITATIONS = [], []
if MART:
    prefix = "[ИМИТАЦИЯ — не вывод] " if DATA_ORIGIN.startswith("ИМИТАЦИЯ") else ""
    top, low = STATS.iloc[0], STATS.iloc[-1]
    overlap = not (top["ДИ 95% низ"] > low["ДИ 95% верх"])
    FINDINGS.append(prefix + (
        f"Тематика и популярность: медиана звёзд максимальна в теме «{top['тема']}» ({top['медиана звёзд']:,.0f}, "
        f"95% ДИ {top['ДИ 95% низ']:,.0f}–{top['ДИ 95% верх']:,.0f}) и минимальна в «{low['тема']}» "
        f"({low['медиана звёзд']:,.0f}, ДИ {low['ДИ 95% низ']:,.0f}–{low['ДИ 95% верх']:,.0f}) — различие в "
        f"{top['медиана звёзд'] / max(low['медиана звёзд'], 1):.1f} раза; "
        + ("интервалы не пересекаются, различие устойчиво к случайности выборки." if not overlap
           else "интервалы пересекаются, поэтому различие между крайними темами статистически не подтверждено.")))
    rho_all = spearman(py.drop_duplicates("repo_ref")["size_kb"], py.drop_duplicates("repo_ref")["stars"])
    p_all = perm_p(py.drop_duplicates("repo_ref")["size_kb"], py.drop_duplicates("repo_ref")["stars"], n=1000)
    rmin, rmax = STATS["ρ(размер, звёзды)"].min(), STATS["ρ(размер, звёзды)"].max()
    strength = "слабая" if abs(rho_all) < 0.3 else ("умеренная" if abs(rho_all) < 0.5 else "сильная")
    FINDINGS.append(prefix + (
        f"Размер и популярность: по уникальным Python-репозиториям ρ Спирмена = {rho_all:+.2f} "
        f"(перестановочный p ≈ {p_all:.3f}) — {strength} {'положительная' if rho_all > 0 else 'отрицательная'} связь"
        + ("" if p_all < 0.05 else ", статистически не подтверждённая") +
        f"; внутри тем ρ от {rmin:+.2f} до {rmax:+.2f}, то есть связь размера с популярностью зависит от тематики."))
    act = STATS.sort_values("push ≤90 дн, %", ascending=False)
    big = py[py["size_band"] == "≥100 МБ"]["active_90d"].mean() * 100
    small = py[py["size_band"] == "<1 МБ"]["active_90d"].mean() * 100
    FINDINGS.append(prefix + (
        f"Активность: доля репозиториев с push за 90 дней — от {act.iloc[-1]['push ≤90 дн, %']:.0f}% («{act.iloc[-1]['тема']}») "
        f"до {act.iloc[0]['push ≤90 дн, %']:.0f}% («{act.iloc[0]['тема']}»); среди репозиториев ≥100 МБ — "
        f"{big:.0f}%, среди <1 МБ — {small:.0f}%."))

    rt_all = MART["repo_topic"]
    multi = rt_all.groupby("repo_ref")["topic"].nunique()
    LIMITATIONS += [
        "Охват: в каждой теме берутся первые 100 репозиториев выдачи «Most stars» с фильтром l=python — выборка смещена "
        "к популярным проектам (усечение снизу), поэтому корреляции занижены по сравнению со всей совокупностью.",
        f"Период: снимки с {py['api_collected_at'].min():%Y-%m-%d} по {py['api_collected_at'].max():%Y-%m-%d}; "
        "звёзды — накопленная величина, «молодые» проекты в невыгодном положении (частично учтено показателем звёзд в год).",
        f"Пересечение тем: {int((multi > 1).sum())} репозиториев входят в несколько тем и учитываются в каждой.",
        f"Пропуски: subscribers — {100 * rt_all['subscribers'].isna().mean():.1f}%"
        + (" (поля нет в ответах Search API — режим без токена)" if rt_all['subscribers'].isna().any() else "")
        + ", "
        f"license — {100 * rt_all['license_spdx'].isna().mean():.1f}%.",
        "size — размер git-репозитория с историей (КБ), а не объём кода; language — основной язык по GitHub Linguist.",
        "Звёзды и push-активность — косвенные меры популярности и поддержки; связь не означает причинности.",
    ]
for i, f in enumerate(FINDINGS, 1):
    print(f"Вывод {i}. {f}\n")
print("Ограничения:")
for x in LIMITATIONS:
    print(" -", x)

Вывод 1. Тематика и популярность: медиана звёзд максимальна в теме «machine-learning» (16,549, 95% ДИ 14,457–19,532) и минимальна в «data-engineering» (428, ДИ 319–652) — различие в 38.7 раза; интервалы не пересекаются, различие устойчиво к случайности выборки.

Вывод 2. Размер и популярность: по уникальным Python-репозиториям ρ Спирмена = +0.47 (перестановочный p ≈ 0.001) — умеренная положительная связь; внутри тем ρ от +0.08 до +0.36, то есть связь размера с популярностью зависит от тематики.

Вывод 3. Активность: доля репозиториев с push за 90 дней — от 40% («telegram») до 67% («cli»); среди репозиториев ≥100 МБ — 79%, среди <1 МБ — 35%.

Ограничения:
 - Охват: в каждой теме берутся первые 100 репозиториев выдачи «Most stars» с фильтром l=python — выборка смещена к популярным проектам (усечение снизу), поэтому корреляции занижены по сравнению со всей совокупностью.
 - Период: снимки с 2026-10-15 по 2026-10-15; звёзды — накопленная величина, «молодые» проекты в невыгодном положении (

## 9. Отчёт о прогоне и оповещения

Отчёт строится из структурного журнала `logs/run_<id>.jsonl` (`src/observability.py`). Пороги оповещения — `config.yaml → alerts` (7 порогов, включая сравнение объёма с прошлым прогоном того же охвата).

In [20]:
REPORT = latest_json("run_report_*.json")
if REPORT is None:
    print("Отчётов о прогоне ещё нет.")
else:
    print(observability.format_report(REPORT))
    req = REPORT["requests"]
    display(pd.Series({"запросов": req["total"], "коды": req["by_status"], "доля ошибок, %": req["error_rate_pct"],
                       "доля 429/403, %": req["rate_limited_pct"], "повторов": req["retries"], "ответов 304": req["not_modified_304"],
                       "латентность медиана, мс": req["latency_ms"]["median"], "латентность p95, мс": req["latency_ms"]["p95"],
                       "трафик тела, Б": req["traffic"]["body_bytes"], "трафик по сети, Б": req["traffic"]["wire_bytes"],
                       "длительность, с": REPORT["duration_s"]}, name="значение").to_frame())
    reports = sorted(DIRS["reports"].glob("run_report_*.json"))
    hist = []
    for p in reports:
        r = json.loads(p.read_text(encoding="utf-8"))
        qd = (r["requests"].get("api_quota") or {}).get("used_delta_by_status", {})
        hist.append({"прогон": r["run_id"], "статус": r["status"], "вердикт": r["verdict"], "код": r["exit_code"],
                     "запросов": r["requests"]["total"], "304": r["requests"]["not_modified_304"],
                     "квота израсходована на 200": qd.get("200"), "на 304": qd.get("304"),
                     "трафик по сети, КБ": round((r["requests"]["traffic"]["wire_bytes"] or 0) / 1024, 1),
                     "длительность, с": r["duration_s"], "оповещений": len(r.get("alerts", []))})
    display(pd.DataFrame(hist))
    print("Колонка «на 304» показывает расход основной квоты GitHub на ответах 304 (по X-RateLimit-Used):")
    print("по документации GitHub 304 на авторизованный запрос квоту не расходует — проверяется этим столбцом.")

Прогон 20261015T215039Z — вердикт: GREEN, код возврата 0
  длительность: 3.38 c; запросов: 3; коды: {'200': 2, '304': 1}
  ошибки: 0.0%; 403/429: 0.0%; повторов: 0; 304: 1 из 2 условных
  латентность: медиана 726.6 мс, p95 1127.1 мс; трафик: тело 659290 Б, по сети 1411 Б (размер известен для 1 ответов: у chunked-ответов GitHub его нет)
  HTML: страниц 1 (+0 из прерванного прогона), карточек 20; API: 200 — 0, 304 — 1, пропущено по водяному знаку — 19, 404 — 0
  чистый слой: репозиториев 592, карточек тем 600, отбраковано 0
  UPSERT repos: +0 / обновлено 0 / без изменений 592; listings: +0 / 1 / 599
  качество: green; mart обновлён
  квота API: остаток 3551 из 5000; расход по кодам ответа: {}


,значение
запросов,3
коды,"{'200': 2, '304': 1}"
"доля ошибок, %",0.0
"доля 429/403, %",0.0
повторов,0
ответов 304,1
"латентность медиана, мс",726.6
"латентность p95, мс",1127.1
"трафик тела, Б",659290
"трафик по сети, Б",1411


,прогон,статус,вердикт,код,запросов,304,квота израсходована на 200,на 304,"трафик по сети, КБ","длительность, с",оповещений
0,20261015T212239Z,interrupted,red,130,122,0,90.0,NaN,1.4,149.80,0
1,20261015T212517Z,interrupted,red,130,255,0,253.0,NaN,1.4,169.81,0
2,20261015T212811Z,finished,green,0,248,0,246.0,NaN,1.4,164.35,0
3,20261015T213102Z,finished,green,0,41,0,9.0,NaN,1.4,97.86,0
4,20261015T213248Z,interrupted,red,130,144,0,112.0,NaN,1.4,169.82,0
5,20261015T214236Z,interrupted,red,130,256,0,254.0,NaN,1.4,169.81,0
6,20261015T214611Z,finished,green,0,225,208,16.0,0.0,1.4,148.84,0
7,20261015T214857Z,finished,green,0,22,17,3.0,0.0,1.4,13.99,0
8,20261015T214919Z,finished,red,3,7,0,NaN,NaN,1.4,5.99,3
9,20261015T214925Z,interrupted,red,130,12,7,1.0,0.0,1.4,15.65,0


Колонка «на 304» показывает расход основной квоты GitHub на ответах 304 (по X-RateLimit-Used):
по документации GitHub 304 на авторизованный запрос квоту не расходует — проверяется этим столбцом.


## 10. Единая точка входа

`src/run_pipeline.py` — запускается одной командой, читает `config.yaml`, пишет журнал и отчёт, возвращает ненулевой код при красном статусе. Ниже — та же команда; по умолчанию из блокнота пайплайн **не запускается** (чтобы повторное исполнение блокнота не нагружало GitHub).

In [21]:
import subprocess
RUN_PIPELINE_FROM_NOTEBOOK = False
CMD = [sys.executable, "src/run_pipeline.py", "--topics", "web-scraping", "--max-pages", "1", "--refresh-max-age-hours", "0"]
print("Команда быстрого инкрементального прогона:", " ".join(CMD[1:]))
if RUN_PIPELINE_FROM_NOTEBOOK:
    res = subprocess.run(CMD, cwd=ROOT, capture_output=True, text=True, timeout=600)
    print(res.stdout[-3000:]); print("Код возврата:", res.returncode)

Команда быстрого инкрементального прогона: src/run_pipeline.py --topics web-scraping --max-pages 1 --refresh-max-age-hours 0


## 11. No-code компонент

Конфигурация B: n8n оркестрирует — `Schedule Trigger` → `Execute Command` (пайплайн) → `Code` (разбор RUN_SUMMARY) → `IF` → оповещение (файл + Telegram) → `Stop and Error` при красном; отдельная ветка `Error Trigger`. Экспорт — `nocode/workflow.json`, сравнение — `nocode/notes.md`.

In [22]:
NOCODE_MAPPING = pd.DataFrame([
    {"этап": "расписание", "в коде": "cron/launchd (запасной вариант)", "в no-code": "Schedule Trigger 17 3 * * * (UTC)", "перенос": "перенесено полностью"},
    {"этап": "запуск пайплайна", "в коде": "src/run_pipeline.py", "в no-code": "Execute Command", "перенос": "n8n только запускает; сбор остаётся в коде"},
    {"этап": "сбор источника", "в коде": "requests + BeautifulSoup, REST API", "в no-code": "—", "перенос": "не переносится: ETag-состояние, водяной знак, robots RFC 9309, детектор блокировки"},
    {"этап": "повторы при ошибке", "в коде": "classify + backoff + Retry-After", "в no-code": "Retry On Fail (2 попытки, 5 с) на узле запуска", "перенос": "частично: в n8n нет Retry-After и классификации кодов"},
    {"этап": "контроль скорости", "в коде": "token bucket на домен + джиттер", "в no-code": "—", "перенос": "не нужен: n8n не обращается к GitHub"},
    {"этап": "контракт данных", "в коде": "pydantic (src/contract.py)", "в no-code": "—", "перенос": "не переносится (строгой типизации нет)"},
    {"этап": "метрики качества", "в коде": "src/quality.py, 9 метрик", "в no-code": "IF по вердикту из RUN_SUMMARY", "перенос": "в n8n только реакция на вердикт"},
    {"этап": "оповещение о сбое", "в коде": "alerts в отчёте о прогоне", "в no-code": "IF → файл alert_*.json + Telegram; Error Trigger; Stop and Error", "перенос": "перенесено: доставка оповещений — сильная сторона n8n"},
])
last_req = (REPORT or {}).get("requests", {}).get("total")
NOCODE_COST = {
    "запусков в сутки": 1,
    "запросов/операций за запуск": f"узлов n8n: ~8; HTTP-запросов пайплайна: {last_req if last_req is not None else '⟪ЗАПОЛНИТЬ после прогона⟫'}",
    "операций в месяц": "~30 выполнений workflow (≈240 выполнений узлов)",
    "тариф и лимит платформы": "n8n self-hosted (npx n8n) — без платы и без лимита выполнений; облачный тариф не используется",
    "вывод о целесообразности": "оркестрация в n8n оправдана (расписание, журнал выполнений, оповещения из коробки); сбор и качество — в коде",
}
display(NOCODE_MAPPING)
pd.Series(NOCODE_COST, name="значение").to_frame()

,этап,в коде,в no-code,перенос
0,расписание,cron/launchd (запасной вариант),Schedule Trigger 17 3 * * * (UTC),перенесено полностью
1,запуск пайплайна,src/run_pipeline.py,Execute Command,n8n только запускает; сбор остаётся в коде
2,сбор источника,"requests + BeautifulSoup, REST API",—,"не переносится: ETag-состояние, водяной знак, robots RFC 9309, детектор блок..."
3,повторы при ошибке,classify + backoff + Retry-After,"Retry On Fail (2 попытки, 5 с) на узле запуска",частично: в n8n нет Retry-After и классификации кодов
4,контроль скорости,token bucket на домен + джиттер,—,не нужен: n8n не обращается к GitHub
5,контракт данных,pydantic (src/contract.py),—,не переносится (строгой типизации нет)
6,метрики качества,"src/quality.py, 9 метрик",IF по вердикту из RUN_SUMMARY,в n8n только реакция на вердикт
7,оповещение о сбое,alerts в отчёте о прогоне,IF → файл alert_*.json + Telegram; Error Trigger; Stop and Error,перенесено: доставка оповещений — сильная сторона n8n


,значение
запусков в сутки,1
запросов/операций за запуск,узлов n8n: ~8; HTTP-запросов пайплайна: 3
операций в месяц,~30 выполнений workflow (≈240 выполнений узлов)
тариф и лимит платформы,n8n self-hosted (npx n8n) — без платы и без лимита выполнений; облачный тари...
вывод о целесообразности,"оркестрация в n8n оправдана (расписание, журнал выполнений, оповещения из ко..."


## 12. Самопроверка готовности проекта

In [23]:
import inspect
def implemented(fn):
    return "raise NotImplementedError" not in inspect.getsource(fn)

CHECKS = [
    ("Паспорт проекта заполнен", lambda: CARD_OK),
    ("Источники описаны", lambda: all(v not in (None, "") for s in SOURCES.values() for v in s.values())),
    ("robots.txt проверен (RFC 9309 + urllib)", lambda: implemented(http_client.PoliteClient.check_robots)),
    ("Диагностика необходимости браузера", lambda: implemented(collect_html.diagnose_rendering)),
    ("Ограничитель скорости", lambda: implemented(http_client.RateLimiter.acquire)),
    ("Классификация ошибок и backoff", lambda: implemented(http_client.classify) and implemented(http_client.backoff_delay)),
    ("Устойчивый fetch", lambda: implemented(http_client.PoliteClient.get)),
    ("Предохранитель", lambda: implemented(http_client.CircuitBreaker.allow)),
    ("Условные запросы (ETag)", lambda: implemented(http_client.EtagStore.headers_for)),
    ("Сбор HTML-источника", lambda: implemented(collect_html.collect_topics)),
    ("Сбор API-источника", lambda: implemented(collect_api.collect_repos)),
    ("Сырой слой непуст", lambda: any(DIRS["raw"].glob("*.jsonl"))),
    ("Очистка записей", lambda: implemented(clean.build_clean)),
    ("Бизнес-ключ (не URL)", lambda: "repo_key" in contract.RepoRecord.model_fields),
    ("Контракт данных и валидация", lambda: implemented(contract.validate_records)),
    ("Схема БД заполнена", lambda: "PRIMARY KEY" in (ROOT / "db/schema.sql").read_text(encoding="utf-8")),
    ("Идемпотентный UPSERT", lambda: implemented(storage.upsert)),
    ("Метрики качества с порогами (>= 6)", lambda: len(CFG["quality"]["metrics"]) >= 6),
    ("Детектор блокировок", lambda: implemented(collect_html.detect_block)),
    ("Слой mart", lambda: any(DIRS["mart"].glob("*.parquet"))),
    ("Графики (>= 2)", lambda: len(list(DIRS["mart"].glob("fig*.png"))) >= 2),
    ("Выводы по данным", lambda: len(FINDINGS) >= 2 and not DATA_ORIGIN.startswith("ИМИТАЦИЯ")),
    ("Ограничения работы указаны", lambda: len(LIMITATIONS) >= 2),
    ("Пороги оповещения (>= 4)", lambda: len(CFG["alerts"]) >= 4),
    ("Структурный журнал пишется", lambda: any(DIRS["logs"].glob("run_*.jsonl"))),
    ("Отчёт о прогоне сформирован", lambda: any(DIRS["reports"].glob("run_report_*.json"))),
    ("Сопоставление код / no-code", lambda: "TODO" not in NOCODE_MAPPING.to_string()),
    ("Оценка стоимости no-code", lambda: all(v is not None for v in NOCODE_COST.values())),
    ("Экспорт n8n workflow", lambda: (ROOT / "nocode/workflow.json").exists()),
    ("Скриншоты n8n", lambda: len(list((ROOT / "nocode/screenshots").glob("*.png"))) >= 3),
]
rows = []
for name, test in CHECKS:
    try:
        ok = bool(test())
    except Exception as e:          # самопроверка не должна падать: ошибка = пункт не выполнен
        ok = False
    rows.append({"пункт": name, "статус": "готово" if ok else "не выполнено"})
readiness = pd.DataFrame(rows)
done = (readiness["статус"] == "готово").sum()
print(f"Выполнено {done} из {len(readiness)} пунктов ({100 * done // len(readiness)}%)")
readiness

Выполнено 29 из 30 пунктов (96%)


,пункт,статус
0,Паспорт проекта заполнен,готово
1,Источники описаны,готово
2,robots.txt проверен (RFC 9309 + urllib),готово
3,Диагностика необходимости браузера,готово
4,Ограничитель скорости,готово
5,Классификация ошибок и backoff,готово
6,Устойчивый fetch,готово
7,Предохранитель,готово
8,Условные запросы (ETag),готово
9,Сбор HTML-источника,готово


## 13. Правовой чек-лист

Полная версия — `docs/legal_checklist.md` (12 пунктов занятия 7).

In [24]:
p = ROOT / "docs" / "legal_checklist.md"
display(Markdown(p.read_text(encoding="utf-8"))) if p.exists() else print("docs/legal_checklist.md не найден")

# Правовой чек-лист (12 пунктов занятия 7)

Проект: «Как популярность и активность Python-репозиториев связаны с их тематикой и размером?»
Автор и ответственный за сбор: Подольский Ф. А. (ПМИ-3-2025). Дата проверки: 15.10.2026, повторная — при реальном прогоне.

| # | Пункт | Ответ |
|---|---|---|
| 1 | **Источники и их назначение** | 1) `https://github.com/topics/<topic>?l=python` — публичные страницы тем (HTML): какие Python-репозитории относятся к теме и в каком порядке; 2) `https://api.github.com/repos/{owner}/{repo}` (без токена — `/search/repositories`) — публичные метрики репозиториев. Оба источника — из согласованного списка курса (связка № 5). |
| 2 | **Цель сбора и её правомерность** | Учебное исследование связи популярности и активности открытых Python-проектов с тематикой и размером. Собираются только публичные сведения о репозиториях; результаты — агрегаты по темам, без оценки отдельных людей. Коммерческого использования и перепродажи данных нет. |
| 3 | **Условия использования: есть ли запрет автоматического сбора** | GitHub Acceptable Use Policies, разд. 7 «Information Usage Restrictions»: использование информации сервиса допускается, в том числе исследователям — для исследований на публичных неперсональных данных при условии открытого доступа к публикациям; запрещено использование для спама, продажи персональных данных, рекрутинга. Под «скрейпингом» GitHub понимает автоматическое извлечение со страниц; сбор через API скрейпингом не считается — поэтому основной объём (метрики) берётся из API, а HTML ограничен 30 страницами тем за прогон. API: GitHub Terms of Service и документация REST API (соблюдение лимитов, запрет обхода ограничений). ⟪ЗАПОЛНИТЬ: дата повторной сверки с текущей редакцией AUP⟫ |
| 4 | **robots.txt: правила, Crawl-delay, can_fetch** | Наш User-Agent `PSU-edu-crawler/1.0` попадает в группу `User-agent: *` (отдельные группы для ИИ-краулеров, Bytespider, AdsBot, bingbot, baidu к нам не относятся). `/topics/<topic>?l=python&page=N` — **разрешено**; `/owner/repo` — разрешено, но не запрашивается; запрещены, в частности, `/*/*/stargazers`, `/*/*/contributors`, `/*/tree/`, `/*?tab=*`, `/*?q=*`, `/search$`. `Crawl-delay` для `*` не задан, `Sitemap` нет. Проверка — `PoliteClient.check_robots` на каждом прогоне (собственный матчер RFC 9309; `urllib.robotparser` в Python 3.10–3.12 не понимает шаблоны `*`, поэтому сравнивается рядом). Копия файла — `raw/robots/github.com_<дата>.txt`. При недоступности robots.txt (5xx/сеть) HTML-сбор в прогоне не выполняется (RFC 9309). |
| 5 | **Используется ли доступ из-под авторизации** | HTML — без входа в аккаунт (страницы тем публичны). API — персональный токен только для повышения квоты (5000 вместо 60 запр/ч) и бесплатных ответов 304; токен без прав доступа, читает лишь публичные данные; хранится в `.env` (вне git), в журналах — маска. Закрытые данные не запрашиваются. |
| 6 | **Применяются ли средства обхода защиты** | **Нет.** Ни прокси, ни ротации IP/User-Agent, ни решения капчи, ни имитации браузера. При признаках блокировки (403/429, «abuse detection», 0 карточек) домен останавливается предохранителем, пишется инцидент, сбор переносится на следующий прогон. |
| 7 | **Объём и частота запросов, пиковая нагрузка на домен** | 1 прогон в сутки. github.com: ≤ 31 запрос за прогон (robots.txt + 30 страниц), темп ≤ 0,33 запр/с с джиттером 30 %. api.github.com: ≤ ~560 запросов за прогон (на практике меньше: водяной знак + 304), темп ≤ 1,5 запр/с (≈ 90/мин) — ниже вторичных лимитов GitHub; при остатке квоты ≤ 50 сбор API останавливается. Предохранитель `max_requests_per_run = 1500`. Фактические числа — в `reports/run_report_*.json`: ⟪ЗАПОЛНИТЬ после прогона⟫. |
| 8 | **Состав данных; есть ли персональные данные** | Метрики репозиториев (звёзды, форки, наблюдатели, issues, размер, даты, лицензия, язык, темы, описание). Из сведений о владельце — только `login` (часть публичного ключа `owner/repo`) и `type` (User/Organization); имена, e-mail, аватары, профили, списки звёздочек и участников **не собираются** (минимизация в `collect_api.minimize`). Логин пользователя может быть псевдонимом конкретного человека, поэтому он считается потенциально идентифицирующим: в выводах используется только агрегированно по темам, рейтингов людей не строится. |
| 9 | **Правовое основание обработки** | Исследовательская цель: п. 9 ч. 1 ст. 6 152-ФЗ (обработка в статистических или иных исследовательских целях при условии обязательного обезличивания) — аналитический слой `mart/` обезличен: вместо `owner/repo` хранится хэш `repo_ref`, логин и полное имя удалены; связь с логином остаётся только в рабочих слоях `raw/`/`clean/`/БД на срок из п. 10, необходимый для соединения источников и UPSERT. GDPR: ст. 6(1)(f) (законный интерес исследования), ст. 89 (гарантии для исследований: минимизация, псевдонимизация); принципы ст. 5 — ограничение целью, минимизация (только login/type), ограничение хранения. |
| 10 | **Место и срок хранения, порядок удаления** | Локально на ноутбуке автора (`raw/`, `clean/`, `db/warehouse.db`), без публикации сырых данных. Срок: до защиты и проверки работы, не дольше 90 дней после неё; затем удаление `raw/`, `clean/`, `db/`, `logs/` (`mart/` с агрегатами по темам может храниться дольше). Токен отзывается после защиты. |
| 11 | **Контакт для связи в User-Agent** | `PSU-edu-crawler/1.0 (+https://github.com/Meta1rus/web-data-extraction; student lab)` — ссылка ведёт на репозиторий курса с описанием сборщика и контактом. |
| 12 | **Действия при получении претензии или блокировки** | 1) немедленно остановить расписание в n8n (деактивировать workflow); 2) не обходить блокировку, не менять IP/UA; 3) зафиксировать инцидент (`logs/incidents/`), ответ сервера сохраняется; 4) связаться с GitHub через `https://support.github.com` (контакт из robots.txt) и с преподавателем; 5) по требованию — удалить собранные данные затронутых репозиториев (по `repo_key` из всех слоёв) и подтвердить удаление; 6) перейти на легальную альтернативу — только API или открытый датасет (например, GH Archive). |


## 14. Паспорт данных (data card)

`docs/data_card.md`: происхождение каждого поля, единицы, частота обновления, покрытие и пропуски, ограничения,
условия использования источника, контакт.

## 15. Подготовка к защите

**Регламент:** 7 минут доклад + 5 минут вопросы. Слайды — `slides.md` (5–7 слайдов).

1. **Инкрементальный прогон (30–60 с):** `bash demo/demo_incremental.sh` — одна тема, одна страница, перепроверка
   20 репозиториев условными запросами → ответы `304`, UPSERT возвращает нули, печатается отчёт о прогоне.
2. **Устойчивость:** `bash demo/demo_failure.sh` — API по неверному адресу → повторы → предохранитель → инцидент
   в `logs/incidents/` → код возврата 3; HTML-часть и mart продолжают работать на ранее собранных метриках.
3. **Возобновляемость (запасной сценарий):** `bash demo/demo_resume.sh` — обрыв после N запросов и продолжение.

Резерв: логи `logs/`, отчёты `reports/`, скриншоты `nocode/screenshots/` с реального прогона.

## 16. Перенос кода в проект

| Функция шаблона | Где в проекте |
|---|---|
| `check_robots` | `src/http_client.py`: `PoliteClient.check_robots`, `RobotsRules` (RFC 9309) |
| `diagnose_rendering` | `src/collect_html.py`: `diagnose_rendering`, CLI `--diagnose` |
| `RateLimiter`, `classify`, `backoff_delay`, `fetch`, `CircuitBreaker`, `conditional_headers` | `src/http_client.py` (`PoliteClient.get`, `EtagStore`) |
| `write_raw`, `read_raw`, `collect_html` | `src/collect_html.py` (`collect_topics`, `parse_topic_page`, `detect_block`) |
| `collect_api` | `src/collect_api.py` (`collect_repos`, `select_keys`, водяной знак) |
| `clean_record`, `business_key` | `src/clean.py` (`listing_from_raw`, `repo_from_raw`, `build_clean`) |
| `Item`, `validate_records` | `src/contract.py` (`TopicListing`, `RepoRecord`) |
| `SCHEMA_SQL`, `init_db`, `upsert_items`, `build_mart` | `db/schema.sql`, `src/storage.py` |
| `quality_report` | `src/quality.py` |
| `log_event`, `run_report`, `alerts` | `src/observability.py` |
| `run_pipeline` | `src/run_pipeline.py` |